# 🚨 Alerting

**Get notified when things go wrong**

## 📋 Overview

**What you'll learn:**
- Alert rules
- Notification channels
- Alert fatigue prevention
- On-call best practices

**Time estimate:** ⏱️ 40 minutes | **Difficulty:** 🟡 Intermediate

## 🚨 Alert Rules

### 1. Error Rate Alert
```yaml
# Prometheus alert rule
groups:
  - name: llm_alerts
    rules:
      - alert: HighErrorRate
        expr: |
          rate(llm_requests_total{status="error"}[5m])
          / rate(llm_requests_total[5m]) > 0.05
        for: 5m
        labels:
          severity: warning
        annotations:
          summary: "High LLM error rate"
          description: "Error rate is {{ $value }}%"
```

### 2. Latency Alert
```yaml
- alert: HighLatency
  expr: |
    histogram_quantile(0.95,
      rate(llm_latency_seconds_bucket[5m])
    ) > 3
  for: 10m
  labels:
    severity: warning
  annotations:
    summary: "High LLM latency"
    description: "P95 latency is {{ $value }}s"
```

### 3. Cost Alert
```yaml
- alert: HighCost
  expr: |
    rate(llm_cost_dollars[1h]) > 100
  for: 5m
  labels:
    severity: critical
  annotations:
    summary: "High LLM costs"
    description: "Spending ${{ $value }}/hour"
```

## 📱 Notification Channels

### Slack Integration
```python
import requests

def send_slack_alert(message: str, severity: str):
    webhook_url = os.getenv('SLACK_WEBHOOK')
    
    color = {
        'critical': 'danger',
        'warning': 'warning',
        'info': 'good'
    }.get(severity, 'warning')
    
    payload = {
        'attachments': [{
            'color': color,
            'title': f'{severity.upper()} Alert',
            'text': message,
            'ts': int(time.time())
        }]
    }
    
    requests.post(webhook_url, json=payload)
```

### PagerDuty Integration
```python
import pypd

pypd.api_key = os.getenv('PAGERDUTY_API_KEY')

def trigger_incident(title: str, description: str):
    pypd.Event.create(
        data={
            'routing_key': os.getenv('PAGERDUTY_ROUTING_KEY'),
            'event_action': 'trigger',
            'payload': {
                'summary': title,
                'severity': 'critical',
                'source': 'llm-api',
                'custom_details': {
                    'description': description
                }
            }
        }
    )
```

## ✅ Summary

**Alert best practices:**
- Only alert on actionable issues
- Set appropriate thresholds
- Use severity levels correctly
- Prevent alert fatigue
- Document runbooks

**Common LLM alerts:**
- Error rate > 5%
- P95 latency > 3s
- Cost > budget
- Cache hit rate < 50%
- API quota near limit

### Next: `11_observability/06_dashboards.ipynb`